*Big Data Assignment - Spark Data Analysis*

Overview
This notebook implements a series of Spark-based data analysis tasks using PySpark to explore real-world datasets such as: Bakery Transactions (sales and weekday revenue analysis)
Restaurant Inspection Data (risk categorization, capacity trends, and ZIP-based risk pivots)
Durham County Foreclosures (spatial and temporal trend analysis)
Global Population Dataset (regional interpolation and growth estimation)

Each section corresponds to a question from the Assignment 2 instructions PDF and includes:

Structured data cleaning and transformation
Appropriate use of Spark SQL functions
Aggregations, joins, pivots, and visualization-ready summaries
Collaboration Note
This assignment was independently authored by Sharayu Rasal,
with some assistance and code review provided in collaboration with ChatGPT
to refine logic, ensure correctness, and align formatting with rubric expectations.

Technologies Used
Apache Spark (PySpark)
JSON and CSV data sources
Spark SQL, DataFrames, and UDFs
Jupyter Notebook

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, dayofweek, date_format, hour, udf, count, sum as _sum,
    explode, lit, size, array, asc, desc, split
)
from pyspark.sql.types import StringType
from pyspark.sql import Window

#Initialize Spark Session
spark = SparkSession.builder \
    .appName("BigData_Assignment2") \
    .getOrCreate()

#Verify Spark is running
spark

#Load datasets
bakery = spark.read.option("header", "true").csv("/home/jovyan/shared/data/bakery.csv")
restaurants = spark.read.option("multiline", "true").json("/home/jovyan/shared/data/Restaurants_in_Durham_County_NC.json")
foreclosures = spark.read.option("multiline", "true").json("/home/jovyan/shared/data/durham-nc-foreclosure-2006-2016.json")
population = spark.read.option("header", "true").csv("/home/jovyan/shared/data/populationbycountry19802010millions.csv")

print("Bakery rows:", bakery.count())
print("Restaurants rows:", restaurants.count())
print("Foreclosures rows:", foreclosures.count())
print("Population rows:", population.count())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/29 23:06:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Bakery rows: 21293
Restaurants rows: 2463
Foreclosures rows: 1948
Population rows: 232


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import countDistinct, count_distinct  # Explicit import
from pyspark.sql.types import *
from pyspark.sql.window import Window
from itertools import combinations

# Helper function to resolve paths
def resolve_path(base_dir, filenames):
    """Find the first existing file from a list of possible filenames"""
    for filename in filenames:
        full_path = os.path.join(base_dir, filename)
        if os.path.exists(full_path):
            return full_path
    return os.path.join(base_dir, filenames[0])

# Define paths
SHARED = "/home/jovyan/shared/data"
PATH_BAKERY = resolve_path(SHARED, ["Bakery.csv", "bakery.csv"])
PATH_RESTAURANTS = resolve_path(SHARED, ["Restaurants_in_Durham_County_NC.json"])
PATH_FORECLOSE = resolve_path(SHARED, ["durham-nc-foreclosure-2006-2016.json"])
PATH_POP = resolve_path(SHARED, ["populationbycountry19802010millions.csv"])

# Initialize Spark session
spark = SparkSession.builder.appName("BigDataAssignment2").getOrCreate()

print("Setup complete!")


Setup complete!


25/10/29 23:07:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# Problem 1: Estimate daily revenue for weekdays by price categories

# Load bakery data
bakery_df = spark.read.csv(PATH_BAKERY, header=True, inferSchema=True)

# Parse date and get day of week
bakery_with_dates = bakery_df.withColumn("date_parsed", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("day_name", date_format(col("date_parsed"), "EEEE"))

# Filter weekdays only
weekday_data = bakery_with_dates.filter(col("day_name").isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]))

# Define price lookup
def get_price(item):
    premium_items = ["Coffee", "Hot chocolate", "Medialuna", "Scandinavian"]
    standard_items = ["Bread", "Pastry", "Muffin", "Cake", "Sandwich"]
    budget_items = ["Tea", "Juice", "Toast", "Cookies"]
    
    if item in premium_items:
        return 5.0
    elif item in standard_items:
        return 3.0
    elif item in budget_items:
        return 2.0
    else:
        return 2.5

price_lookup_udf = udf(get_price, DoubleType())

# Apply pricing
weekday_priced = weekday_data.withColumn("item_price", price_lookup_udf(col("Item")))

# Calculate daily revenue - Using count_distinct instead
daily_stats = weekday_priced.groupBy("date_parsed", "day_name") \
    .agg(
        count_distinct("Transaction").alias("Total_Transactions"),
        sum("item_price").alias("Estimated_Revenue")
    )

# Format and show top 10
top_revenue_days = daily_stats.orderBy(col("Estimated_Revenue").desc()).limit(10) \
    .withColumn("Estimated_Revenue", concat(lit("$"), format_number(col("Estimated_Revenue"), 2))) \
    .select(
        col("date_parsed").alias("Date"),
        col("day_name").alias("Weekday"),
        "Total_Transactions",
        "Estimated_Revenue"
    )

print("Problem 1: Top 10 Revenue Days")
top_revenue_days.show(10, truncate=False)


Problem 1: Top 10 Revenue Days


+----------+---------+------------------+-----------------+
|Date      |Weekday  |Total_Transactions|Estimated_Revenue|
+----------+---------+------------------+-----------------+
|2016-10-31|Monday   |96                |$684.00          |
|2016-11-04|Friday   |85                |$657.00          |
|2016-11-03|Thursday |95                |$636.50          |
|2016-11-11|Friday   |93                |$608.50          |
|2017-02-17|Friday   |63                |$581.00          |
|2016-11-18|Friday   |77                |$570.00          |
|2016-12-29|Thursday |61                |$552.00          |
|2016-11-07|Monday   |75                |$545.00          |
|2016-11-02|Wednesday|83                |$541.50          |
|2016-12-19|Monday   |76                |$539.50          |
+----------+---------+------------------+-----------------+



In [4]:
# Problem 2: Geographic zones with most high-capacity, moderate-risk restaurants

# Load restaurant data with multiline option
restaurant_data = spark.read.option("multiline", True).json(PATH_RESTAURANTS)

# Extract and rename the fields we need
restaurants = restaurant_data.select(
    col("fields.est_group_desc").alias("establishment_type"),
    col("fields.rpt_area_desc").alias("area_description"),
    col("fields.status").alias("status"),
    col("fields.seats").alias("seats"),
    col("fields.risk").alias("risk_level"),
    col("fields.geolocation").alias("location")
)

# Apply filters and create zones
filtered_data = restaurants \
    .filter((col("status") == "ACTIVE") & (col("area_description") == "Food Service")) \
    .filter(col("establishment_type").isNotNull() & col("seats").isNotNull() & (col("seats") >= 20)) \
    .filter(col("risk_level").isin([3, 4])) \
    .filter(size("location") == 2) \
    .withColumn("latitude", element_at("location", 1)) \
    .filter(col("latitude").isNotNull()) \
    .withColumn("zone",
                when(col("latitude") >= 36.00, "North")
                .when((col("latitude") >= 35.95) & (col("latitude") < 36.00), "Central")
                .otherwise("South"))

# Group by zone and establishment type
zone_results = filtered_data.groupBy("zone", "establishment_type") \
    .agg(
        count("*").alias("restaurant_count"),
        sum(col("seats")).cast("long").alias("total_seats")
    ) \
    .orderBy(desc("restaurant_count"), desc("total_seats")) \
    .limit(15) \
    .select(
        col("zone").alias("Zone"),
        col("establishment_type").alias("Establishment_Type"),
        col("restaurant_count").alias("Restaurant_Count"),
        col("total_seats").alias("Total_Seats")
    )

print("Problem 2: Top 15 Zone-Establishment Combinations")
zone_results.show(15, truncate=False)

Problem 2: Top 15 Zone-Establishment Combinations


[Stage 25:>                                                         (0 + 1) / 1]

+-------+-----------------------+----------------+-----------+
|Zone   |Establishment_Type     |Restaurant_Count|Total_Seats|
+-------+-----------------------+----------------+-----------+
|Central|Full-Service Restaurant|92              |13179      |
|South  |Full-Service Restaurant|84              |10491      |
|North  |Full-Service Restaurant|76              |8496       |
|North  |Fast Food Restaurant   |25              |1637       |
|South  |Fast Food Restaurant   |20              |1287       |
|Central|Fast Food Restaurant   |18              |3058       |
|North  |Elementary School      |2               |200        |
|South  |Hospital               |2               |186        |
|Central|Elementary School      |2               |92         |
|North  |Hospital               |1               |400        |
|North  |Nursing Home           |1               |350        |
|South  |Nursing Home           |1               |102        |
|South  |Elementary School      |1               |54   

In [5]:
# Problem 3: Top selling item per hour

# Load bakery data
bakery_data = spark.read.option("header", True).option("inferSchema", True).csv(PATH_BAKERY)

# Cast Time to timestamp and extract hour
bakery_with_time = bakery_data \
    .withColumn("time_stamp", col("Time").cast("timestamp")) \
    .filter(col("time_stamp").isNotNull() & col("Item").isNotNull()) \
    .withColumn("hour", hour("time_stamp")) \
    .filter((col("hour") >= 6) & (col("hour") <= 21))

# Create UDF to categorize time periods
@udf(returnType=StringType())
def get_time_category(h):
    if h is None:
        return None
    if 6 <= h <= 10:
        return "Morning"
    if 11 <= h <= 15:
        return "Afternoon"
    if 16 <= h <= 23:
        return "Evening"
    return "Other"

# Add time category
bakery_categorized = bakery_with_time.withColumn("time_category", get_time_category(col("hour")))

# Count occurrences per hour and item
item_counts = bakery_categorized.groupBy("hour", "Item").agg(count("*").alias("count"))

# Rank items within each hour
hour_window = Window.partitionBy("hour").orderBy(desc("count"), asc("Item"))
top_per_hour = item_counts \
    .withColumn("rank", row_number().over(hour_window)) \
    .filter((col("rank") == 1) & (col("hour") >= 7) & (col("hour") <= 21)) \
    .drop("rank") \
    .join(bakery_categorized.select("hour", "time_category").distinct(), on="hour", how="left") \
    .select("hour", "Item", "count", "time_category") \
    .orderBy("hour")

print("Problem 3 - Output 1: Top item per hour (7-21)")
top_per_hour.show(50, truncate=False)

# Create pivot table for hours 7-21
hours_list = list(range(7, 22))
item_hour_counts = bakery_categorized \
    .filter((col("hour") >= 7) & (col("hour") <= 21)) \
    .groupBy("Item", "hour") \
    .agg(count("*").alias("count"))

# Pivot with specific hours
pivot_table = item_hour_counts \
    .groupBy("Item") \
    .pivot("hour", hours_list) \
    .sum("count") \
    .fillna(0)

# Add total column by adding each hour column
total_expression = col(str(hours_list[0]))
for h in hours_list[1:]:
    total_expression = total_expression + col(str(h))

pivot_with_total = pivot_table \
    .withColumn("Total", total_expression) \
    .orderBy(desc("Total"))

print("\nProblem 3 - Output 2: Pivot table (Items by Hours 7-21)")
pivot_with_total.select(["Item"] + [str(h) for h in hours_list]).show(50, truncate=False)

Problem 3 - Output 1: Top item per hour (7-21)


+----+------------------------+-----+-------------+
|hour|Item                    |count|time_category|
+----+------------------------+-----+-------------+
|7   |Coffee                  |13   |Morning      |
|8   |Coffee                  |199  |Morning      |
|9   |Coffee                  |583  |Morning      |
|10  |Coffee                  |820  |Morning      |
|11  |Coffee                  |946  |Afternoon    |
|12  |Coffee                  |740  |Afternoon    |
|13  |Coffee                  |607  |Afternoon    |
|14  |Coffee                  |636  |Afternoon    |
|15  |Coffee                  |519  |Afternoon    |
|16  |Coffee                  |321  |Evening      |
|17  |Coffee                  |69   |Evening      |
|18  |Afternoon with the baker|14   |Evening      |
|19  |Tshirt                  |11   |Evening      |
|20  |Postcard                |7    |Evening      |
|21  |Hot chocolate           |2    |Evening      |
+----+------------------------+-----+-------------+


Problem 3 

[Stage 41:>                                                         (0 + 1) / 1]

+------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|Item                    |7  |8  |9  |10 |11 |12 |13 |14 |15 |16 |17 |18 |19 |20 |21 |
+------------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|Coffee                  |13 |199|583|820|946|740|607|636|519|321|69 |11 |6  |1  |0  |
|Bread                   |2  |171|400|508|528|474|340|341|310|196|46 |6  |2  |0  |0  |
|Tea                     |0  |21 |103|156|176|183|181|233|207|126|41 |5  |3  |0  |0  |
|Cake                    |0  |8  |40 |83 |133|127|124|182|174|124|30 |0  |0  |0  |0  |
|Pastry                  |2  |57 |191|203|151|97 |48 |49 |32 |16 |8  |2  |0  |0  |0  |
|NONE                    |1  |24 |79 |97 |114|167|159|96 |32 |13 |4  |0  |0  |0  |0  |
|Sandwich                |0  |5  |17 |20 |53 |162|234|171|75 |29 |4  |1  |0  |0  |0  |
|Medialuna               |6  |43 |120|125|108|55 |36 |48 |34 |25 |11 |4  |1  |0  |0  |
|Hot chocolate           |0  |9  |56 |76 |9

In [6]:
# Problem 4: Foreclosure Trends by Neighborhood and Crisis Year Buckets

# Load foreclosure data
foreclosure_data = spark.read.json(PATH_FORECLOSE)

# Extract necessary fields from the nested structure
foreclosure_parsed = foreclosure_data.select(
    col("fields.parcel_number").alias("parcel_number"),
    col("fields.geocode").alias("geocode"),
    col("fields.year").alias("year_field")
)

# Extract year from the year field (might be timestamp or string)
foreclosure_with_year = foreclosure_parsed.withColumn(
    "year_value",
    year(col("year_field"))
)

# Create year buckets based on crisis periods
foreclosure_bucketed = foreclosure_with_year.withColumn(
    "year_bucket",
    when((col("year_value") >= 2006) & (col("year_value") <= 2009), "2006-2009")
    .when((col("year_value") >= 2010) & (col("year_value") <= 2013), "2010-2013")
    .when((col("year_value") >= 2014) & (col("year_value") <= 2016), "2014-2016")
    .otherwise(None)
)

# Extract area code (first 3 characters of parcel number)
foreclosure_with_area = foreclosure_bucketed.withColumn(
    "area_code",
    substring(col("parcel_number"), 1, 3)
)

# Filter out null year buckets
foreclosure_clean = foreclosure_with_area.filter(col("year_bucket").isNotNull())

# Calculate total foreclosures per area code
area_totals = foreclosure_clean.groupBy("area_code") \
    .agg(count("*").alias("total_foreclosures"))

# Count foreclosures per area and year bucket
area_year_counts = foreclosure_clean.groupBy("area_code", "year_bucket") \
    .agg(count("*").alias("count"))

# Pivot year buckets into columns
pivot_by_year = area_year_counts.groupBy("area_code") \
    .pivot("year_bucket", ["2006-2009", "2010-2013", "2014-2016"]) \
    .sum("count") \
    .fillna(0)

# Join with total foreclosures
final_results = pivot_by_year.join(area_totals, "area_code") \
    .filter(col("total_foreclosures") >= 50) \
    .select(
        col("area_code").alias("Area_Code"),
        col("total_foreclosures").alias("Total_Foreclosures"),
        col("2006-2009"),
        col("2010-2013"),
        col("2014-2016")
    ) \
    .orderBy(desc("Total_Foreclosures")) \
    .limit(10)

print("Problem 4: Top 10 Area Codes by Foreclosures (≥50 total)")
final_results.show(10, truncate=False)


Problem 4: Top 10 Area Codes by Foreclosures (≥50 total)
+---------+------------------+---------+---------+---------+
|Area_Code|Total_Foreclosures|2006-2009|2010-2013|2014-2016|
+---------+------------------+---------+---------+---------+
|112      |147               |97       |36       |14       |
|159      |142               |5        |65       |72       |
|118      |136               |111      |16       |9        |
|111      |80                |56       |17       |7        |
|117      |70                |35       |21       |14       |
|130      |63                |19       |37       |7        |
|110      |62                |48       |12       |2        |
|116      |62                |36       |23       |3        |
|119      |62                |46       |12       |4        |
|114      |58                |40       |12       |6        |
+---------+------------------+---------+---------+---------+



In [7]:
# Problem 5: Analyze item co-purchase patterns on weekdays by daypart

from itertools import combinations

# Load bakery data
bakery_data = spark.read.option("header", True).option("inferSchema", True).csv(PATH_BAKERY)

# Parse date and time
bakery_parsed = bakery_data \
    .withColumn("date_value", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("time_stamp", col("Time").cast("timestamp")) \
    .withColumn("weekday_name", date_format(col("date_value"), "EEEE")) \
    .withColumn("hour_value", hour(col("time_stamp")))

# Filter for weekdays only
weekday_data = bakery_parsed.filter(
    col("weekday_name").isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"])
)

# Define day part based on hour
weekday_with_daypart = weekday_data.withColumn(
    "day_part",
    when((col("hour_value") >= 6) & (col("hour_value") <= 10), "Breakfast")
    .when((col("hour_value") >= 11) & (col("hour_value") <= 15), "Lunch")
    .when((col("hour_value") >= 16) & (col("hour_value") <= 23), "Dinner")
    .otherwise(None)
).filter(col("day_part").isNotNull())

# Group items by transaction
transaction_items = weekday_with_daypart.groupBy("date_value", "Transaction", "day_part") \
    .agg(collect_list("Item").alias("items"))

# Filter for multi-item transactions (more than 1 item)
multi_item_transactions = transaction_items.filter(size(col("items")) > 1)

# Create UDF to generate all unique pairs from a list of items
def generate_item_pairs(items_list):
    if items_list is None or len(items_list) < 2:
        return []
    # Get unique items and sort for consistency
    unique_items = sorted(set(items_list))
    # Generate all combinations of 2 items
    pairs = list(combinations(unique_items, 2))
    return [{"item1": pair[0], "item2": pair[1]} for pair in pairs]

pair_schema = ArrayType(StructType([
    StructField("item1", StringType()),
    StructField("item2", StringType())
]))

generate_pairs_udf = udf(generate_item_pairs, pair_schema)

# Generate pairs for each transaction
transaction_pairs = multi_item_transactions \
    .withColumn("pairs", generate_pairs_udf(col("items"))) \
    .select("day_part", explode("pairs").alias("pair")) \
    .select("day_part", col("pair.item1").alias("item1"), col("pair.item2").alias("item2"))

# Count pair occurrences per day part
pair_counts = transaction_pairs.groupBy("day_part", "item1", "item2") \
    .agg(count("*").alias("pair_count"))

# Rank pairs within each day part
daypart_window = Window.partitionBy("day_part").orderBy(desc("pair_count"))
ranked_pairs = pair_counts.withColumn("rank", row_number().over(daypart_window))

# Get top 5 pairs per day part
top_pairs = ranked_pairs.filter(col("rank") <= 5) \
    .select(
        col("day_part").alias("DayPart"),
        col("item1").alias("Item1"),
        col("item2").alias("Item2"),
        col("pair_count").alias("PairCount")
    ) \
    .orderBy("DayPart", desc("PairCount"))

print("Problem 5: Top 5 Item Pairs by Day Part")
top_pairs.show(50, truncate=False)


Problem 5: Top 5 Item Pairs by Day Part


[Stage 59:>                                                         (0 + 1) / 1]

+---------+------+---------+---------+
|DayPart  |Item1 |Item2    |PairCount|
+---------+------+---------+---------+
|Breakfast|Coffee|Pastry   |158      |
|Breakfast|Bread |Coffee   |153      |
|Breakfast|Bread |Pastry   |90       |
|Breakfast|Coffee|Medialuna|85       |
|Breakfast|Coffee|Toast    |79       |
|Dinner   |Cake  |Coffee   |54       |
|Dinner   |Bread |Coffee   |46       |
|Dinner   |Coffee|Tea      |39       |
|Dinner   |Coffee|Cookies  |32       |
|Dinner   |Cake  |Tea      |28       |
|Lunch    |Bread |Coffee   |320      |
|Lunch    |Cake  |Coffee   |224      |
|Lunch    |Coffee|Sandwich |209      |
|Lunch    |Coffee|Tea      |197      |
|Lunch    |Coffee|Pastry   |138      |
+---------+------+---------+---------+



In [8]:
# Problem 6: Analyze restaurant risk patterns by ZIP code

# Load restaurant data
restaurant_data = spark.read.option("multiline", True).json(PATH_RESTAURANTS)

# Filter for ACTIVE Food Service restaurants
active_restaurants = restaurant_data.select(
    col("fields.premise_zip").alias("zip_code"),
    col("fields.status").alias("status"),
    col("fields.rpt_area_desc").alias("area_type"),
    col("fields.risk").alias("risk_level"),
    col("fields.geolocation").alias("location")
).filter(
    (col("status") == "ACTIVE") & (col("area_type") == "Food Service")
)

# Flatten geolocation and filter for valid coordinates (exactly 2 elements)
restaurants_with_coords = active_restaurants \
    .filter(size("location") == 2) \
    .withColumn("latitude", element_at("location", 1)) \
    .withColumn("longitude", element_at("location", 2)) \
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())

# Create UDF to categorize risk levels
def categorize_risk(risk_value):
    if risk_value is None:
        return None
    if risk_value == 0:
        return "No Risk"
    elif risk_value in [1, 2]:
        return "Low Risk"
    elif risk_value == 3:
        return "Medium Risk"
    elif risk_value == 4:
        return "High Risk"
    else:
        return None

risk_category_udf = udf(categorize_risk, StringType())

# Apply risk categorization
restaurants_categorized = restaurants_with_coords.withColumn(
    "risk_category",
    risk_category_udf(col("risk_level"))
).filter(col("risk_category").isNotNull())

# Count restaurants per ZIP and risk category
zip_risk_counts = restaurants_categorized.groupBy("zip_code", "risk_category") \
    .agg(count("*").alias("count"))

print("Problem 6 - Output 1: Count by ZIP and risk category")
zip_risk_counts.select(
    col("zip_code").alias("zip"),
    col("risk_category").alias("risk_category"),
    "count"
).orderBy("zip_code", "risk_category").show(50, truncate=False)

# Find ZIPs with both Low Risk AND High Risk restaurants
zips_low_risk = zip_risk_counts.filter(col("risk_category") == "Low Risk") \
    .select("zip_code").distinct()

zips_high_risk = zip_risk_counts.filter(col("risk_category") == "High Risk") \
    .select("zip_code").distinct()

zips_both_levels = zips_low_risk.intersect(zips_high_risk).orderBy("zip_code")

print("\nProblem 6 - Output 2: ZIPs with both Low Risk and High Risk")
zips_both_levels.select(col("zip_code").alias("zip")).show(50, truncate=False)

# Create pivot table: ZIP codes as rows, risk categories as columns
risk_categories = ["No Risk", "Low Risk", "Medium Risk", "High Risk"]
pivot_risk_table = zip_risk_counts.groupBy("zip_code") \
    .pivot("risk_category", risk_categories) \
    .sum("count") \
    .fillna(0)

# Add total column
pivot_with_total = pivot_risk_table \
    .withColumn("total", 
                col("No Risk") + col("Low Risk") + col("Medium Risk") + col("High Risk")) \
    .orderBy(desc("total"))

print("\nProblem 6 - Output 3: Pivot table (ZIP by Risk Categories)")
pivot_with_total.select(
    col("zip_code").alias("zip"),
    "No Risk",
    "Low Risk",
    "Medium Risk",
    "High Risk",
    "total"
).show(50, truncate=False)

Problem 6 - Output 1: Count by ZIP and risk category


+----------+-------------+-----+
|zip       |risk_category|count|
+----------+-------------+-----+
|27503     |High Risk    |2    |
|27514     |High Risk    |2    |
|27514     |Low Risk     |1    |
|27514     |Medium Risk  |2    |
|27517     |High Risk    |7    |
|27517     |Low Risk     |5    |
|27560     |High Risk    |3    |
|27560     |Low Risk     |2    |
|27560     |Medium Risk  |2    |
|27560-6815|High Risk    |1    |
|27572     |Low Risk     |1    |
|27603     |High Risk    |2    |
|27701     |High Risk    |86   |
|27701     |Low Risk     |87   |
|27701     |Medium Risk  |18   |
|27703     |High Risk    |56   |
|27703     |Low Risk     |56   |
|27703     |Medium Risk  |23   |
|27704     |High Risk    |44   |
|27704     |Low Risk     |37   |
|27704     |Medium Risk  |15   |
|27705     |High Risk    |81   |
|27705     |Low Risk     |69   |
|27705     |Medium Risk  |30   |
|27706     |High Risk    |1    |
|27707     |High Risk    |85   |
|27707     |Low Risk     |82   |
|27707    

+-----+
|zip  |
+-----+
|27514|
|27517|
|27560|
|27701|
|27703|
|27704|
|27705|
|27707|
|27708|
|27709|
|27710|
|27712|
|27713|
+-----+


Problem 6 - Output 3: Pivot table (ZIP by Risk Categories)


[Stage 77:>                                                         (0 + 1) / 1]

+----------+-------+--------+-----------+---------+-----+
|zip       |No Risk|Low Risk|Medium Risk|High Risk|total|
+----------+-------+--------+-----------+---------+-----+
|27707     |0      |82      |35         |85       |202  |
|27701     |0      |87      |18         |86       |191  |
|27705     |0      |69      |30         |81       |180  |
|27713     |0      |67      |14         |79       |160  |
|27703     |0      |56      |23         |56       |135  |
|27704     |0      |37      |15         |44       |96   |
|27708     |0      |11      |1          |25       |37   |
|27712     |0      |10      |7          |13       |30   |
|27709     |0      |1       |0          |16       |17   |
|27517     |0      |5       |0          |7        |12   |
|27560     |0      |2       |2          |3        |7    |
|27710     |0      |4       |0          |3        |7    |
|27514     |0      |1       |2          |2        |5    |
|27603     |0      |0       |0          |2        |2    |
|27503     |0 

In [9]:
# Problem 7: Regional Population Interpolation & Growth Rate Analysis (Extra Credit)

# Load population data
pop_raw = spark.read.option("header", True).csv(PATH_POP)

# Get first column name and rename to Country
first_column = pop_raw.columns[0]
pop_data = pop_raw.withColumnRenamed(first_column, "Country") \
    .withColumn("Country", trim(col("Country"))) \
    .select(["Country"] + [c for c in pop_raw.columns if c != first_column])

# Unpivot years 1980-2010 into long format using stack
years_list = list(map(str, range(1980, 2011)))
stack_expression = ", ".join([f"'{yr}', `{yr}`" for yr in years_list])

long_format = pop_data.select(
    "Country",
    expr(f"stack({len(years_list)}, {stack_expression}) as (year_str, pop_str)")
) \
    .withColumn("year_value", col("year_str").cast("int")) \
    .withColumn("population", expr("try_cast(pop_str as double)")) \
    .drop("year_str", "pop_str")

# Linear interpolation for missing values
prev_window = Window.partitionBy("Country").orderBy("year_value").rowsBetween(Window.unboundedPreceding, 0)
next_window = Window.partitionBy("Country").orderBy(col("year_value").desc()).rowsBetween(Window.unboundedPreceding, 0)

interpolated = long_format \
    .withColumn("year_nonnull", when(col("population").isNotNull(), col("year_value"))) \
    .withColumn("pop_prev", last("population", ignorenulls=True).over(prev_window)) \
    .withColumn("year_prev", last("year_nonnull", ignorenulls=True).over(prev_window)) \
    .withColumn("pop_next", first("population", ignorenulls=True).over(next_window)) \
    .withColumn("year_next", first("year_nonnull", ignorenulls=True).over(next_window)) \
    .withColumn("pop_interpolated",
        when(col("population").isNotNull(), col("population"))
        .when(
            col("pop_prev").isNotNull() & col("pop_next").isNotNull() & (col("year_next") != col("year_prev")),
            col("pop_prev") + (col("pop_next") - col("pop_prev")) * 
            (col("year_value") - col("year_prev")) / (col("year_next") - col("year_prev"))
        )
        .when(col("pop_prev").isNotNull(), col("pop_prev"))
        .otherwise(col("pop_next"))
    )

# Pivot to get decade columns
decades = [1980, 1990, 2000, 2010]
decade_pivot = interpolated.filter(col("year_value").isin(decades)) \
    .groupBy("Country") \
    .pivot("year_value", decades) \
    .agg(first("pop_interpolated"))

# Remove aggregate rows and require valid endpoints
excluded_regions = {
    "World", "Africa", "Asia", "Europe", "Oceania", "European Union", "Euro area",
    "Caribbean", "Central & South America", "Central America", "South America",
    "North America", "Middle East", "Western Europe", "Eastern Europe", "CIS"
}

decade_clean = decade_pivot \
    .filter(~col("Country").isin(list(excluded_regions))) \
    .filter(col("1980").isNotNull() & col("2010").isNotNull())

# Round populations to 3 decimal places
for decade_col in map(str, decades):
    decade_clean = decade_clean.withColumn(decade_col, round(col(decade_col), 3))

# Map countries to regions
asia_countries = {
    "China", "India", "Indonesia", "Pakistan", "Bangladesh", "Japan", "Philippines",
    "Vietnam", "Turkey", "Iran", "Thailand", "Myanmar", "South Korea", "Iraq",
    "Afghanistan", "Saudi Arabia", "Uzbekistan", "Malaysia", "Yemen", "Nepal",
    "North Korea", "Sri Lanka", "Kazakhstan", "Syria", "Cambodia", "Jordan"
}

americas_countries = {
    "United States", "Canada", "Mexico", "Guatemala", "Honduras", "El Salvador",
    "Nicaragua", "Costa Rica", "Panama", "Cuba", "Haiti", "Dominican Republic",
    "Jamaica", "Trinidad and Tobago", "Colombia", "Venezuela", "Ecuador", "Peru",
    "Bolivia", "Chile", "Argentina", "Uruguay", "Paraguay", "Brazil", "Guyana"
}

africa_countries = {
    "Nigeria", "Ethiopia", "Egypt", "South Africa", "Tanzania", "Kenya", "Uganda",
    "Algeria", "Sudan", "Morocco", "Angola", "Ghana", "Mozambique", "Madagascar",
    "Cameroon", "Niger", "Burkina Faso", "Mali", "Malawi", "Zambia", "Senegal",
    "Chad", "Somalia", "Zimbabwe", "Guinea", "Rwanda", "Benin", "Burundi", "Tunisia"
}

europe_countries = {
    "United Kingdom", "France", "Germany", "Italy", "Spain", "Portugal", "Netherlands",
    "Belgium", "Switzerland", "Austria", "Denmark", "Norway", "Sweden", "Finland",
    "Poland", "Czech Republic", "Czechia", "Slovakia", "Hungary", "Romania",
    "Bulgaria", "Greece", "Russia", "Belarus", "Ukraine", "Lithuania", "Latvia"
}

@udf(returnType=StringType())
def assign_region(country_name):
    if country_name in asia_countries:
        return "Asia"
    if country_name in americas_countries:
        return "Americas"
    if country_name in africa_countries:
        return "Africa"
    if country_name in europe_countries:
        return "Europe"
    return None

# Add region and filter
with_regions = decade_clean \
    .withColumn("Region", assign_region(col("Country"))) \
    .filter(col("Region").isNotNull())

# Calculate per-country growth rate
with_growth = with_regions.withColumn(
    "country_growth_rate",
    round(when(col("1980") > 0, (col("2010") / col("1980") - 1) * 100), 1)
)

# Calculate regional average growth (population-weighted)
regional_totals = with_growth.groupBy("Region") \
    .agg(
        sum("1980").alias("total_1980"),
        sum("2010").alias("total_2010")
    ) \
    .withColumn(
        "regional_avg_growth",
        round((col("total_2010") / col("total_1980") - 1) * 100, 1)
    ) \
    .select("Region", "regional_avg_growth")

# Join everything together
final_output = with_growth.join(regional_totals, on="Region", how="left") \
    .select(
        "Region",
        "Country",
        "1980",
        "1990",
        "2000",
        "2010",
        col("regional_avg_growth").alias("Avg_Growth_Rate")  # Only keep regional growth
    ) \
    .orderBy("Region", "Country")

print("Problem 7: Country Populations by Decade with Growth Rates")
final_output.show(100, truncate=False)

Problem 7: Country Populations by Decade with Growth Rates


25/10/29 23:10:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/10/29 23:10:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010
 Schema: _c0, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010
Expected: _c0 but found: 
CSV file: file:///home/jovyan/shared/data/populationbycountry19802010millions.csv
[Stage 91:>                                                         (0 + 1) / 1]

+--------+-------------------+-------+--------+--------+--------+---------------+
|Region  |Country            |1980   |1990    |2000    |2010    |Avg_Growth_Rate|
+--------+-------------------+-------+--------+--------+--------+---------------+
|Africa  |Algeria            |18.806 |25.089  |30.429  |34.586  |107.7          |
|Africa  |Angola             |6.743  |8.297   |10.377  |13.068  |107.7          |
|Africa  |Benin              |3.458  |4.705   |6.619   |9.056   |107.7          |
|Africa  |Burkina Faso       |6.318  |8.361   |11.588  |16.242  |107.7          |
|Africa  |Burundi            |4.298  |5.536   |6.823   |9.863   |107.7          |
|Africa  |Cameroon           |8.762  |11.885  |15.343  |19.294  |107.7          |
|Africa  |Chad               |4.522  |5.841   |7.943   |10.543  |107.7          |
|Africa  |Egypt              |42.634 |54.907  |65.159  |80.472  |107.7          |
|Africa  |Ethiopia           |38.605 |51.535  |64.165  |88.013  |107.7          |
|Africa  |Ghana 